# 03 — Code Generation Demo (Playwright Version)

This notebook converts the resolved mapping output (from Notebook 06)
into deterministic Playwright test code.

Pipeline so far:
NL → IR → Mapping → **Code Generation** → Execution → Artifacts → Analysis

This notebook focuses on:
- Loading resolved selectors
- Generating Playwright test code
- Ensuring deterministic, stable output


In [7]:
import json
from pathlib import Path


## 3.1 — Load Mapping Output

We load the resolved mapping file produced by Notebook 06.



In [8]:
mapping_path = Path("../sample-data/mapping_output/login_mapping.json")

with open(mapping_path, "r") as f:
    resolved_steps = json.load(f)

resolved_steps


FileNotFoundError: [Errno 2] No such file or directory: '../sample-data/mapping_output/login_mapping.json'

## 3.2 — Playwright Code Templates

We define simple templates for:
- imports
- browser setup
- action generation
- teardown

These templates keep the generated code deterministic.


In [ ]:
HEADER = """\
from playwright.async_api import async_playwright, expect

async def run_test():
    async with async_playwright() as pw:
        browser = await pw.chromium.launch(headless=True)
        page = await browser.new_page()
        await page.goto("https://demo.playwright.dev/todomvc")
"""

FOOTER = """\
        await browser.close()

if __name__ == "__main__":
    import asyncio
    asyncio.run(run_test())
"""


## 3.3 — Action Generator

This function converts each IR step into Playwright code using the
resolved selectors from Notebook 06.


In [ ]:
def generate_action(step):
    action = step["action"]
    resolved = step["resolved"]

    if "error" in resolved:
        return f"# ERROR: unresolved target '{step['logical_target']}'"

    selector = resolved["selector"]

    if action == "enter_text":
        text = step.get("value", "")
        return f'        await page.fill("{selector}", "{text}")'

    if action == "press_enter":
        return f'        await page.keyboard.press("Enter")'

    if action == "click":
        return f'        await page.click("{selector}")'

    if action == "assert_visible":
        return f'        await expect(page.locator("{selector}")).to_be_visible()'

    if action == "assert_count":
        count = step.get("value", 1)
        return f'        await expect(page.locator("{selector}")).to_have_count({count})'

    return f"# TODO: unsupported action '{action}'"


## 3.4 — Generate Full Playwright Test Code

We combine:
- header
- generated actions
- footer

into a single deterministic Python script.


In [ ]:
generated_lines = [HEADER]

for step in resolved_steps:
    generated_lines.append(generate_action(step))

generated_lines.append(FOOTER)

generated_code = "\n".join(generated_lines)
print(generated_code)


## 3.5 — Save Generated Test Code

Notebook 04 will execute this file directly.


In [ ]:
output_path = Path("../generated-tests/todomvc_test.py")
output_path.parent.mkdir(parents=True, exist_ok=True)

with open(output_path, "w") as f:
    f.write(generated_code)

output_path


## Summary

In this notebook we:

- Loaded resolved selectors from Notebook 06
- Defined Playwright code templates
- Converted IR steps into deterministic Playwright actions
- Generated a complete runnable test script
- Saved the script for Notebook 04 to execute

Next step: run the generated test in Notebook 04.
